In [ ]:
#Tarefa 1.8

import pandas as pd
import numpy as np
import matplotlib as plt

In [ ]:
print('\n Carregando os arquivos necessarios')

df_orders = pd.read_csv(r'C:\GameStoreBrasil\output\orders_cleaned.csv', parse_dates=['order_date'] )
df_games = pd.read_csv(r'C:\GameStoreBrasil\data\games.csv', parse_dates=['release_date'])
df_members = pd.read_csv(r'C:\GameStoreBrasil\output\members_cleaned.csv', parse_dates=['join_date'])

In [ ]:
print(f' -Orders_cleaned.csv: {len(df_orders)} linhas')
print(f' -games.csv: {len(df_games)} linhas')

In [ ]:
print("Filtrando valores negativos")
linhas_antes = len(df_orders)
df_orders = df_orders[(df_orders['quantity']>0) & (df_orders['unit_price']>0)]
linhas_removidas = linhas_antes - len(df_orders)

print(f' -Linhas Removidas:{linhas_removidas}')
print(f' -Linhas Restantes:{len(df_orders)}')


In [ ]:
df_orders['revenue'] = df_orders['unit_price'] * df_orders['quantity']



In [ ]:
df_merged = df_orders.merge(df_games[['game_id', 'game_name', 'platform', 'price', 'cost']], on = 'game_id', how = 'inner')

print(f"linhas apos o merge {len(df_merged)}")

game_performance = df_merged.groupby(['game_id', 'game_name', 'platform']).agg(
    avg_selling_price = ('unit_price', 'mean'), #Preco medio de venda
    catallog_price = ("price",'first'), #Preco de catalogo(Primeiro valor)
    total_quantity = ('quantity', 'sum'), #Quantidade total Vendida
    total_revenue = ("revenue",'sum'),  #Receita total
    num_transactions = ('order_id', 'count'), #Numero de transacoes
    cost = ('cost', 'sum')
).reset_index()

game_performance['profit_margin'] = (game_performance.total_revenue - game_performance.total_quantity * game_performance.cost)/game_performance.total_revenue

print(game_performance.head())
print(f'\n -Jogos unicos com vendas :{len(game_performance)}')

price_std = df_merged.groupby('game_id')['unit_price'].std().reset_index()
price_std.columns = ['game_id', 'price_std']
game_performance = game_performance.merge(price_std, on='game_id', how = 'left')

game_performance['price_std'] = game_performance['price_std'].fillna(0)


In [ ]:
print("\n Primeiras 10 linhas da performance por jogo")

print(game_performance.head(10))

print(game_performance[['avg_selling_price', 'total_quantity', 'price_std']].describe())

jogos_com_variacao = (game_performance['price_std']>0).sum()

print(f'\n - Jogos com variacao de preco (std>0): {jogos_com_variacao}')
jogos_sem_variacao = (game_performance['price_std']==0).sum()

In [ ]:
perf = df_merged.groupby(['game_id']).agg(
    total_quantity_sold = ('quantity', 'sum'),
    total_revenue = ('revenue', 'sum')
).reset_index()

perf = perf.merge(df_games[['game_id','price', 'cost']], on = 'game_id')
perf['profit_margin'] = (perf.total_revenue - perf.total_quantity_sold * perf.cost)/perf.total_revenue

perf = perf.sort_values("total_revenue", ascending=False)

perf.head()